# Data0 and the frozen gene split
Only data0 is accessed. All reported scores are validation estimates.

In [1]:
from pathlib import Path
import os
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
import json
import pandas as pd
from IPython.display import display, Image
from m6a_project.training import load_config, run_experiment
from m6a_project.tracking import rebuild_index
from m6a_project.cache import prepare_data
CONFIG = load_config("configs/experiments/baselines_v1.toml")


In [2]:
cache = prepare_data(CONFIG)
manifest = cache["manifest"]
display(pd.DataFrame(cache["identity"]["summary"]).T)
print("Split SHA-256:", cache["identity"]["split_hash"])

,sites,genes,positive_sites,prevalence
train,96488.0,3077.0,4372.0,0.045311
val,25350.0,775.0,1103.0,0.043511


Split SHA-256: cbf2d0405abc707a557802dd3bc87cd218aae5a9b597ee71d1a58665dd8c07ad


In [3]:
from m6a_project.splitting import assert_disjoint
assert_disjoint(manifest)
display(manifest.groupby("partition").n_reads.describe())

,count,mean,std,min,25%,50%,75%,max
partition,,,,,,,,
train,96488.0,88.492963,133.032937,20.0,31.0,47.0,83.0,991.0
val,25350.0,98.169507,152.357004,20.0,32.0,50.0,87.0,988.0


In [4]:
from m6a_project.embeddings import SequenceEmbedding
embedding = SequenceEmbedding()
display(embedding.summary())
display(pd.DataFrame(embedding.values, index=embedding.motifs, columns=["PC1", "PC2"]).head(12))

{'method': 'positional_nucleotide_onehot_then_PCA',
 'vocabulary_size': 66,
 'explained_variance_ratio': [0.2644508832474579, 0.20176073210795645],
 'distinct_vectors_at_10_decimals': 35}

,PC1,PC2
AAAAC,0.459394,-0.254434
AAACA,-0.682263,0.911241
AAACC,-0.105185,0.617819
AAACT,-0.682263,0.911241
AACAA,-0.720688,-0.741052
AACAC,-0.143610,-1.034474
AACAG,-0.684379,-0.950918
AACAT,-0.720688,-0.741052
AACCA,-1.285267,0.131200
AACCC,-0.708189,-0.162222


The PCA vocabulary comes from motif grammar, not validation frequencies or labels. This fixed representation differs from m6Anet's learned neural embedding. Read models inherit site labels, which are noisy read targets.